<a href="https://colab.research.google.com/github/gundalarakesh262-cpu/nifty100-financial-intelligence/blob/main/notebooks/Ratio_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import sqlite3

In [5]:
conn = sqlite3.connect("nifty100.db")

In [6]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name


In [7]:
companies = pd.read_sql("SELECT * FROM companies", conn)
profit = pd.read_sql("SELECT * FROM profitandloss", conn)
balance = pd.read_sql("SELECT * FROM balancesheet", conn)
cashflow = pd.read_sql("SELECT * FROM cashflow", conn)
market = pd.read_sql("SELECT * FROM market_cap", conn)

DatabaseError: Execution failed on sql 'SELECT * FROM companies': no such table: companies

In [ ]:
companies.head()
profit.head()
balance.head()
cashflow.head()

In [ ]:
print(profit.columns)
print(balance.columns)
print(cashflow.columns)

In [ ]:
import re
import numpy as np
import pandas as pd


def safe_divide(numerator, denominator):
    """
    Safely divide two columns.
    If denominator is 0 or missing, return NaN instead of crashing.
    """
    return np.where(
        (denominator == 0) | pd.isna(denominator),
        np.nan,
        numerator / denominator
    )


def extract_year(year_value):
    """
    Converts values like 'Mar-23' into 2023.
    """
    year_str = str(year_value)
    match = re.search(r"(\d{2,4})", year_str)

    if not match:
        return np.nan

    year = int(match.group(1))

    if year < 100:
        return 2000 + year

    return year


In [ ]:

# Create fiscal_year column so we can join with market_cap table
profit["fiscal_year"] = profit["year"].apply(extract_year)
balance["fiscal_year"] = balance["year"].apply(extract_year)
cashflow["fiscal_year"] = cashflow["year"].apply(extract_year)

# Merge main financial statements
df = (
    profit
    .merge(balance, on=["company_id", "year", "fiscal_year"], how="inner", suffixes=("_pl", "_bs"))
    .merge(cashflow, on=["company_id", "year", "fiscal_year"], how="left")
)

# Merge company data for face value
df = df.merge(
    companies[["id", "face_value"]],
    left_on="company_id",
    right_on="id",
    how="left"
)

# Merge market cap / valuation data
df = df.merge(
    market,
    left_on=["company_id", "fiscal_year"],
    right_on=["company_id", "year"],
    how="left",
    suffixes=("", "_market")
)

df.head()



In [ ]:
ratios = pd.DataFrame()

ratios["company_id"] = df["company_id"]
ratios["year"] = df["year"]
ratios["fiscal_year"] = df["fiscal_year"]

equity = df["equity_capital"] + df["reserves"]
capital_employed = equity + df["borrowings"]
ebit = df["operating_profit"] - df["depreciation"]
ebitda = df["operating_profit"]

# -----------------------------
# 1. Profitability KPIs
# -----------------------------
ratios["net_profit_margin_pct"] = safe_divide(df["net_profit"], df["sales"]) * 100
ratios["operating_profit_margin_pct"] = safe_divide(df["operating_profit"], df["sales"]) * 100
ratios["pbt_margin_pct"] = safe_divide(df["profit_before_tax"], df["sales"]) * 100
ratios["ebit_margin_pct"] = safe_divide(ebit, df["sales"]) * 100
ratios["expense_ratio_pct"] = safe_divide(df["expenses"], df["sales"]) * 100
ratios["return_on_equity_pct"] = safe_divide(df["net_profit"], equity) * 100
ratios["return_on_capital_employed_pct"] = safe_divide(ebit, capital_employed) * 100
ratios["return_on_assets_pct"] = safe_divide(df["net_profit"], df["total_assets"]) * 100

# -----------------------------
# 2. Debt / Leverage KPIs
# -----------------------------
ratios["debt_to_equity"] = safe_divide(df["borrowings"], equity)
ratios["debt_to_assets"] = safe_divide(df["borrowings"], df["total_assets"])
ratios["debt_to_capital"] = safe_divide(df["borrowings"], capital_employed)
ratios["liabilities_to_assets"] = safe_divide(df["total_liabilities"], df["total_assets"])
ratios["interest_coverage"] = safe_divide(
    df["operating_profit"] + df["other_income"],
    df["interest"]
)
ratios["net_debt_cr"] = df["borrowings"] - df["investments"]
ratios["net_debt_to_ebitda"] = safe_divide(ratios["net_debt_cr"], ebitda)

# -----------------------------
# 3. Cash Flow KPIs
# -----------------------------
ratios["cash_from_operations_cr"] = df["operating_activity"]
ratios["free_cash_flow_cr"] = df["operating_activity"] + df["investing_activity"]
ratios["capex_cr"] = df["investing_activity"].abs()
ratios["cfo_margin_pct"] = safe_divide(df["operating_activity"], df["sales"]) * 100
ratios["fcf_margin_pct"] = safe_divide(ratios["free_cash_flow_cr"], df["sales"]) * 100
ratios["cfo_to_pat"] = safe_divide(df["operating_activity"], df["net_profit"])
ratios["capex_to_sales_pct"] = safe_divide(ratios["capex_cr"], df["sales"]) * 100
ratios["fcf_to_ebitda"] = safe_divide(ratios["free_cash_flow_cr"], ebitda)

# -----------------------------
# 4. Per Share KPIs
# -----------------------------
share_count = safe_divide(df["equity_capital"], df["face_value"])

ratios["earnings_per_share"] = df["eps"]
ratios["book_value_per_share"] = safe_divide(equity, share_count)
ratios["dividend_payout_ratio_pct"] = df["dividend_payout"]

# -----------------------------
# 5. Valuation KPIs
# -----------------------------
ratios["market_cap_crore"] = df["market_cap_crore"]
ratios["enterprise_value_crore"] = df["enterprise_value_crore"]
ratios["pe_ratio"] = df["pe_ratio"]
ratios["pb_ratio"] = df["pb_ratio"]
ratios["ev_ebitda"] = df["ev_ebitda"]
ratios["dividend_yield_pct"] = df["dividend_yield_pct"]
ratios["earnings_yield_pct"] = safe_divide(1, df["pe_ratio"]) * 100
ratios["fcf_yield_pct"] = safe_divide(ratios["free_cash_flow_cr"], df["market_cap_crore"]) * 100

ratios.head()



In [ ]:
# Save to CSV
ratios.to_csv("../data/processed/financial_ratios_generated.csv", index=False)

# Save to SQLite table
ratios.to_sql("financial_ratios_generated", conn, if_exists="replace", index=False)

print("Ratio Engine Version 1 completed!")
print("Rows:", len(ratios))
print("Columns:", len(ratios.columns))